In [1]:
from selenium import webdriver
from bs4 import BeautifulSoup as bs
from urllib.parse import urljoin
import pandas as pd

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
# pagina web
driver = webdriver.Chrome(service= Service(ChromeDriverManager().install()))
url = "https://books.toscrape.com/"

In [3]:
titulosucio = []
preciosucio = []
stocksucio = []
titulos = []
precios = []
stocks = []

In [5]:
while True:
    driver.get(url)
    contenido = driver.page_source # HTML a lenguaje python, tambien obtenemos la pagina
    soup = bs(contenido)


    #Loop principal el que se encarga de extraer la informacion
    for libro in soup.find_all("article", attrs={"class":"product_pod"}):
        titulo = libro.find("h3")
        titulosucio.append(titulo.text)

        precio = libro.find("p", attrs = {"class":"price_color"})
        preciosucio.append(precio.text)

        stock = libro.find("p",attrs = {"class":"instock availability"})
        stocksucio.append(stock.text)

    # Loops que van limpiando el texto poco a poco mediante va pasando pagina
    for libro in titulosucio:
        titulos.append(libro.replace("\n","").strip())
    for libro in preciosucio:
        precios.append(libro.replace("\n","").strip())
    for libro in stocksucio:
        stocks.append(libro.replace("\n","").strip())

    # Encuentra la siguiente pagina para scrapear, una vez completado el primer loop y consiguientes
    
    siguientepag = soup.select_one("li.next>a")
    if siguientepag:
        siguienteurl = siguientepag.get("href")
        url = urljoin(url,siguienteurl)
    else:
        break


In [6]:
# Ahora almacenamos toda la informacion de datos en un dataframe y lo exportamos en un CSV
df = pd.DataFrame({"Titulo":titulos, "Precio":precios,"Stock":stocks})


In [10]:
df.head(20)

,Titulo,Precio,Stock
0,A Light in the ...,£51.77,In stock
1,Tipping the Velvet,£53.74,In stock
2,Soumission,£50.10,In stock
3,Sharp Objects,£47.82,In stock
4,Sapiens: A Brief History ...,£54.23,In stock
5,The Requiem Red,£22.65,In stock
6,The Dirty Little Secrets ...,£33.34,In stock
7,The Coming Woman: A ...,£17.93,In stock
8,The Boys in the ...,£22.60,In stock
9,The Black Maria,£52.15,In stock


In [11]:
df.to_csv("librosCompleto.csv",index= False,encoding = "utf-8")